In [15]:
import numpy as np
import ase.io

In [16]:
def get_mae(first, second):
    delta = first - second
    return np.mean(np.abs(delta))

def get_mae_norm(first, second):
    first_norm = np.sqrt(np.sum(first ** 2, axis = -1))
    print(first_norm.shape)
    second_norm = np.sqrt(np.sum(second ** 2, axis = -1))
    return get_mae(first_norm, second_norm)

def get_mae_atomic(first, second, n_atoms):
    first_per_atom, second_per_atom = [], []
    for i in range(len(n_atoms)):
        first_per_atom.append(first[i] / n_atoms[i])
        second_per_atom.append(second[i] / n_atoms[i])
    first_per_atom = np.array(first_per_atom)
    second_per_atom = np.array(second_per_atom)
    return get_mae(first_per_atom, second_per_atom)


def get_statistics(values):
    mean = np.mean(values)
    delta = values - mean
    std = np.sqrt(np.mean(delta * delta))
    return np.array([mean, std])

In [17]:
structures = ase.io.read('../../../final_reproduce/datasets/hme21/hme21_test.xyz', index = ':')
energies = [struc.info['energy'] for struc in structures]
energies = np.array(energies)

forces = [struc.arrays['forces'] for struc in structures]
forces = np.concatenate(forces, axis = 0)

n_atoms = [len(struc.positions) for struc in structures]
energies_per_atom = np.array([energies[i] / n_atoms[i] for i in range(len((energies)))])

In [18]:
calcs = ['hme21_ultra_fast_lr_decay_250_0_loss_per_atom/',
         'hme21_ultra_fast_lr_decay_250_1_loss_per_atom/',
         'hme21_ultra_fast_lr_decay_250_2_loss_per_atom/',
         'hme21_ultra_fast_lr_decay_250_3_loss_per_atom/',
         'hme21_ultra_fast_lr_decay_250_4_loss_per_atom/']

def get_predictions(calc):
    chunks = []
    for i in range(8):
        chunks.append(f'{calc}/hme21_predictions_{i}')
        
    predictions_energies, predictions_forces = [], []
    for chunk in chunks:
        predictions_energies.append(np.load(chunk + '/energies_predicted.npy'))
        predictions_forces.append(np.load(chunk + '/forces_predicted.npy'))

    predictions_energies = np.concatenate(predictions_energies, axis = 0)
    predictions_forces = np.concatenate(predictions_forces, axis = 0)
    
    return predictions_energies, predictions_forces

all_predictions_energies, all_predictions_forces = [], []
all_mae_forces_norm, all_mae_atomic = [], []
all_mae_forces = []
for calc in calcs:
    predictions_energies, predictions_forces = get_predictions(calc)
    
    mae_forces_norm = get_mae_norm(forces, predictions_forces)
    mae_forces = get_mae(forces, predictions_forces)
    mae_atomic = get_mae_atomic(energies, predictions_energies, n_atoms)
    
    all_mae_forces_norm.append(mae_forces_norm)
    all_mae_atomic.append(mae_atomic)
    all_mae_forces.append(mae_forces)    
        
    all_predictions_energies.append(predictions_energies[np.newaxis])
    all_predictions_forces.append(predictions_forces[np.newaxis])
    


(69572,)
(69572,)
(69572,)
(69572,)
(69572,)


In [19]:
print("loss per atom:")

print("single model mae forces norm: ", 1000 * get_statistics(all_mae_forces_norm))
print("single model mae forces: ", 1000 * get_statistics(all_mae_forces))
print('single model mae atomic: ', 1000 * get_statistics(all_mae_atomic))

all_predictions_energies = np.concatenate(all_predictions_energies, axis = 0)
all_predictions_forces = np.concatenate(all_predictions_forces, axis = 0)

energies_predicted = np.mean(all_predictions_energies, axis = 0)
forces_predicted = np.mean(all_predictions_forces, axis = 0)

print(energies_predicted.shape, forces_predicted.shape)

mae_forces_norm = get_mae_norm(forces, forces_predicted)
mae_atomic = get_mae_atomic(energies, energies_predicted, n_atoms)
mae_forces = get_mae(forces, forces_predicted)

print('ensemble', mae_forces_norm, mae_atomic, mae_forces)


loss per atom:
single model mae forces norm:  [141.59356312   1.88631687]
single model mae forces:  [125.08205348   1.63468404]
single model mae atomic:  [17.779503    0.14611882]
(2495,) (69572, 3)
(69572,)
ensemble 0.1285159734783102 0.016801468968274284 0.11406008122701562


In [6]:


predictions_energies_per_atom = np.array([predictions_energies[i] / n_atoms[i] for i in range(len(predictions_energies))])


print(energies.shape)
print(forces.shape)

(2495,)
(69572, 3)


In [7]:
def get_mae(first, second):
    delta = first - second
    return np.mean(np.abs(first - second))

def get_norm(vectors):
    squared = vectors * vectors
    return np.sqrt(squared.sum(axis = 1))

In [8]:
print('force norm mae: ', get_mae(get_norm(forces), get_norm(predictions_forces)))
print('energies mae per atom: ', get_mae(energies_per_atom, predictions_energies_per_atom))
print('force mae: ', get_mae(forces, predictions_forces))

force norm mae:  0.14074431597352288
energies mae per atom:  0.017622345383520463
force mae:  0.12443610546778103
